# Well Data Analysis Solution

This notebook demonstrates how to analyze well data based on statistical and depth information. The solution can identify mudstone and sandstone from lithology data and calculate various metrics for different stratigraphic periods.

## Key Features:
1. **Lithology Identification**: Identifies mudstone (泥岩) and sandstone (砂岩) from Chinese text
2. **Metric Calculations**: 
   - Mudstone thickness (cumulative thickness for lithology containing '泥岩')
   - Stratigraphic thickness (difference between bottom and top depths)
   - Mud-to-layer ratio (mudstone thickness / stratigraphic thickness)
   - Average values for TOC, S1, Brittleness index, Ro within depth ranges
3. **Multi-period Analysis**: Supports multiple stratigraphic periods (沙三下, 沙四上晚期, 沙四上中期, 沙四上早期)
4. **Structured Output**: Generates separate tables for each stratigraphic period
5. **Export Capabilities**: Supports Excel and CSV export formats

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from well_analyzer import WellDataAnalyzer, load_sample_data, create_sample_intervals

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## 1. Initialize the Well Data Analyzer

Create an instance of the WellDataAnalyzer class which provides all the necessary functionality for analyzing well data.

In [ ]:
# Create analyzer instance
analyzer = WellDataAnalyzer()

print("Well Data Analyzer initialized successfully!")
print(f"Supported stratigraphic periods: {analyzer.stratigraphic_periods}")

## 2. Load Sample Data

For demonstration purposes, we'll use sample data. In a real scenario, you would load your actual well data from Excel, CSV, or database files.

In [ ]:
# Load sample data
sample_data = load_sample_data()

print(f"Sample data loaded:")
print(f"- Total records: {len(sample_data)}")
print(f"- Number of wells: {sample_data['well_name'].nunique()}")
print(f"- Depth range: {sample_data['depth'].min():.0f}m - {sample_data['depth'].max():.0f}m")

# Display first few rows
print("\nFirst 5 rows of sample data:")
sample_data.head()

## 3. Examine Lithology Distribution

Let's examine the distribution of different lithology types in our sample data.

In [ ]:
# Analyze lithology distribution
lithology_counts = sample_data['lithology'].value_counts()
print("Lithology distribution:")
print(lithology_counts)

# Create visualization
plt.figure(figsize=(10, 6))
lithology_counts.plot(kind='bar', color=['#8B4513', '#F4A460', '#D2691E', '#CD853F', '#A0522D'])
plt.title('Distribution of Lithology Types', fontsize=14, fontweight='bold')
plt.xlabel('Lithology Type')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Test lithology identification
print("\nLithology identification test:")
test_lithologies = ['泥岩', '砂岩', '石灰岩', '泥质砂岩']
for lith in test_lithologies:
    identified = analyzer.identify_lithology(lith)
    print(f"'{lith}' -> {identified}")

## 4. Define Well Intervals

Define the depth intervals for each well and stratigraphic period. This represents the real-world scenario where geologists have identified specific depth ranges for each stratigraphic period in different wells.

In [ ]:
# Create sample intervals
sample_intervals = create_sample_intervals()

print("Well intervals defined:")
for well_name, intervals in sample_intervals.items():
    print(f"\n{well_name}:")
    for period, (top, bottom) in intervals.items():
        print(f"  {period}: {top}m - {bottom}m (厚度: {bottom-top}m)")

## 5. Perform Well Data Analysis

Now we'll run the complete analysis, calculating all metrics for each well and stratigraphic period.

In [ ]:
# Perform analysis
print("Performing well data analysis...")
results = analyzer.analyze_well_data(sample_data, sample_intervals)

print(f"Analysis completed for {len(results)} stratigraphic periods")

# Display summary statistics
for period, df in results.items():
    if not df.empty:
        print(f"\n{period}: {len(df)} wells analyzed")
    else:
        print(f"\n{period}: No data available")

## 6. Display Detailed Results

Let's examine the detailed results for each stratigraphic period.

In [ ]:
# Display detailed results for each period
for period, df in results.items():
    print(f"\n{'='*60}")
    print(f"地层时期: {period}")
    print(f"{'='*60}")
    
    if not df.empty:
        # Select key columns for display
        display_cols = ['well_name', 'top_depth', 'bottom_depth', 'mudstone_thickness', 
                       'stratigraphic_thickness', 'mud_to_layer_ratio', 'avg_TOC', 
                       'avg_S1', 'avg_brittleness_index', 'avg_Ro']
        
        display_df = df[display_cols].copy()
        
        # Round numerical values for better display
        numerical_cols = ['mudstone_thickness', 'stratigraphic_thickness', 'mud_to_layer_ratio',
                         'avg_TOC', 'avg_S1', 'avg_brittleness_index', 'avg_Ro']
        
        for col in numerical_cols:
            display_df[col] = display_df[col].round(3)
        
        print(display_df.to_string(index=False))
        
        # Summary statistics
        print(f"\n总结统计:")
        print(f"- 平均泥岩厚度: {df['mudstone_thickness'].mean():.2f}m")
        print(f"- 平均地层厚度: {df['stratigraphic_thickness'].mean():.2f}m")
        print(f"- 平均泥地比: {df['mud_to_layer_ratio'].mean():.3f}")
        print(f"- 平均TOC: {df['avg_TOC'].mean():.3f}")
        
    else:
        print("此地层时期无数据")

## 7. Generate Summary Tables

Create formatted summary tables with Chinese column names for better presentation.

In [ ]:
# Generate summary tables
summary_tables = analyzer.generate_summary_tables()

print("Summary Tables Generated:")
print("="*80)

for period, table in summary_tables.items():
    print(f"\n地层时期: {period}")
    print("-"*60)
    
    if not table.empty:
        print(table.to_string(index=False))
    else:
        print("此地层时期无数据")
    print()

## 8. Data Visualization

Create visualizations to better understand the data patterns across different stratigraphic periods.

In [ ]:
# Create visualization of mud-to-layer ratio by period
plt.figure(figsize=(15, 10))

# Prepare data for visualization
plot_data = []
for period, df in results.items():
    if not df.empty:
        for _, row in df.iterrows():
            plot_data.append({
                'period': period,
                'well': row['well_name'],
                'mud_ratio': row['mud_to_layer_ratio'],
                'avg_TOC': row['avg_TOC'],
                'mudstone_thickness': row['mudstone_thickness']
            })

if plot_data:
    plot_df = pd.DataFrame(plot_data)
    
    # Subplot 1: Mud-to-layer ratio by period
    plt.subplot(2, 2, 1)
    sns.boxplot(data=plot_df, x='period', y='mud_ratio')
    plt.title('Mud-to-Layer Ratio by Stratigraphic Period')
    plt.xlabel('Stratigraphic Period')
    plt.ylabel('Mud-to-Layer Ratio')
    plt.xticks(rotation=45)
    
    # Subplot 2: Average TOC by period
    plt.subplot(2, 2, 2)
    sns.boxplot(data=plot_df, x='period', y='avg_TOC')
    plt.title('Average TOC by Stratigraphic Period')
    plt.xlabel('Stratigraphic Period')
    plt.ylabel('Average TOC')
    plt.xticks(rotation=45)
    
    # Subplot 3: Mudstone thickness by well
    plt.subplot(2, 2, 3)
    sns.barplot(data=plot_df, x='well', y='mudstone_thickness', hue='period')
    plt.title('Mudstone Thickness by Well and Period')
    plt.xlabel('Well Name')
    plt.ylabel('Mudstone Thickness (m)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Subplot 4: Correlation between TOC and Mud Ratio
    plt.subplot(2, 2, 4)
    sns.scatterplot(data=plot_df, x='avg_TOC', y='mud_ratio', hue='period', size='mudstone_thickness')
    plt.title('TOC vs Mud-to-Layer Ratio')
    plt.xlabel('Average TOC')
    plt.ylabel('Mud-to-Layer Ratio')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("No data available for visualization")

## 9. Export Results

Export the analysis results to Excel and CSV formats for further use.

In [ ]:
# Export results to Excel
try:
    analyzer.export_results('well_analysis_results.xlsx', format='excel')
    print("Results exported to Excel successfully!")
except Exception as e:
    print(f"Error exporting to Excel: {e}")

# Export results to CSV
try:
    analyzer.export_results('well_analysis_results.csv', format='csv')
    print("Results exported to CSV successfully!")
except Exception as e:
    print(f"Error exporting to CSV: {e}")

## 10. Custom Data Input Example

Here's how you would use the analyzer with your own data:

In [ ]:
# Example of how to use with custom data
print("Example: Using the analyzer with custom data")
print("="*50)

# Create example custom data
custom_data = pd.DataFrame({
    'well_name': ['TestWell_A', 'TestWell_A', 'TestWell_A', 'TestWell_B', 'TestWell_B'],
    'depth': [1100, 1150, 1200, 1100, 1150],
    'lithology': ['泥岩', '砂岩', '泥岩', '砂岩', '泥岩'],
    'TOC': [2.5, 1.2, 3.1, 1.0, 2.8],
    'S1': [1.5, 0.8, 1.9, 0.6, 1.7],
    'brittleness_index': [0.4, 0.7, 0.3, 0.8, 0.4],
    'Ro': [0.9, 0.7, 1.1, 0.6, 1.0],
    'thickness': [50, 50, 50, 50, 50]
})

custom_intervals = {
    'TestWell_A': {
        '沙三下': (1050, 1250)
    },
    'TestWell_B': {
        '沙三下': (1050, 1200)
    }
}

# Create new analyzer instance for custom data
custom_analyzer = WellDataAnalyzer()

# Analyze custom data
custom_results = custom_analyzer.analyze_well_data(custom_data, custom_intervals)

# Display custom results
for period, df in custom_results.items():
    if not df.empty:
        print(f"\nCustom analysis results for {period}:")
        print(df[['well_name', 'mudstone_thickness', 'stratigraphic_thickness', 
                 'mud_to_layer_ratio', 'avg_TOC']].to_string(index=False))

print("\nCustom analysis completed!")

## Conclusion

This well data analysis solution provides comprehensive functionality for:

1. **Automated Lithology Identification**: Accurately identifies mudstone and sandstone from Chinese text descriptions
2. **Multi-metric Calculations**: Computes essential petroleum geology metrics including mudstone thickness, stratigraphic thickness, mud-to-layer ratio, and average geochemical properties
3. **Multi-well and Multi-period Analysis**: Handles complex datasets with multiple wells and stratigraphic periods
4. **Flexible Data Input**: Works with various data formats and structures
5. **Professional Output**: Generates formatted tables with Chinese headers and exports to Excel/CSV
6. **Visualization Support**: Includes data visualization capabilities for better insights

The solution is designed to be easily extensible and can be adapted for different geological contexts and requirements.